In [1]:
import os, sys
import yaml
import geopandas as gpd
import pandas as pd
import numpy as np

sys.path.append('/home/dnash/repos/eaton_scripps_CO_ARs/modules')
from plot_trajectory_maps import subset_gdf_to_plot

In [2]:
path_to_data = '/expanse/nfs/cw3e/cwp140/' 
path_to_out  = '../out/'       # output files (numerical results, intermediate datafiles) -- read & write

In [3]:
def create_table_summary_trajectories(ARDT, ssn):
    ar = True
    # Load trajectory GeoJSON data
    gdf = gpd.read_file("/home/dnash/repos/eaton_scripps_CO_ARs/out/trajectories.geojson")
    gdf.crs = 'epsg:3857'
    gdf = gdf.set_index(pd.to_datetime(gdf['start_date']))
    
    # import configuration file for region list of HUC8s
    yaml_doc = '/home/dnash/repos/eaton_scripps_CO_ARs/data/HUC8_regions.yml'
    config = yaml.load(open(yaml_doc), Loader=yaml.SafeLoader)

    region_lst = ['northern_upper_CO', 'southern_upper_CO', 'rio_grande', 'eastern_CO']
    region_lbl = ['Northwestern', 'Southwestern', 'Rio Grande', 'Eastern']
    df_lst = []
    for i, region in enumerate(region_lst):
        ## get number of subbasins per region
        HUC8_lst = config[region]
        nsubbasins = len(HUC8_lst)
        idx = (gdf['region'] == region)
        tmp = gdf.loc[idx]
        total_no_trajs = len(tmp)
        
        subset_gdf = subset_gdf_to_plot(gdf, ARDT, ssn, ar, region=region, basin=None, HUC8=None)
        AR_trajs = len(subset_gdf)
        perc_AR =  round((AR_trajs / total_no_trajs)*100, 1)
        total_AR_trajs = '{0}\%'.format(perc_AR)
    
        ## get number of AR scale
        arscale_txt = []
        for k, arscale in enumerate(np.arange(1, 6)):
            idx = (subset_gdf['ar_scale'] == arscale)
            arscale_trajs = len(subset_gdf.loc[idx])
            perc_ar = round((arscale_trajs / total_no_trajs)*100, 1)
            arscale_txt.append('{0}\%'.format(perc_ar))
    
        # d = {'Region': region_lbl[i], 'No. of Subbasins': nsubbasins, 'Total Trajectories': total_no_trajs,
        #      'Landfalling AR Trajectories': total_AR_trajs, 'AR Scale 1': arscale_txt[0], 'AR Scale 2': arscale_txt[1],
        #      'AR Scale 3': arscale_txt[2], 'AR Scale 4': arscale_txt[3], 'AR Scale 5': arscale_txt[4]}

        d = {'Region': region_lbl[i], 'Total Trajectories': total_no_trajs,
             'Landfalling AR Trajectories': total_AR_trajs, 'AR scale 1': arscale_txt[0], 'AR scale 2': arscale_txt[1],
             'AR scale 3': arscale_txt[2], 'AR scale 4': arscale_txt[3], 'AR scale 5': arscale_txt[4]}
    
        df_lst.append(pd.DataFrame(d, index=[i]))
    
    df = pd.concat(df_lst)
    df = df.T
    # df = df.set_index(['Region'], drop=True)
    df.rename(columns=df.iloc[0], inplace = True)
    df.drop(df.index[0], inplace = True)
    return df

<>:26: SyntaxWarning: invalid escape sequence '\%'
<>:34: SyntaxWarning: invalid escape sequence '\%'
<>:26: SyntaxWarning: invalid escape sequence '\%'
<>:34: SyntaxWarning: invalid escape sequence '\%'
/scratch/dnash/job_39383321/ipykernel_3716098/2324447628.py:26: SyntaxWarning: invalid escape sequence '\%'
  total_AR_trajs = '{0}\%'.format(perc_AR)
/scratch/dnash/job_39383321/ipykernel_3716098/2324447628.py:34: SyntaxWarning: invalid escape sequence '\%'
  arscale_txt.append('{0}\%'.format(perc_ar))


In [4]:
ARDT_lst = ['tARget', 'ar', 'ar_scale']
ssn_lst = ['NDJFMA', 'MJJASO']
ssn_lbl = ['cool', 'warm']
ARDT_lbl = ['tARgetv4', 'Rutz', 'AR scale']
for i, ssn in enumerate(ssn_lst):
    for j, ARDT in enumerate(ARDT_lst):
        table = create_table_summary_trajectories(ARDT, ssn)
        print('% Table for {0} {1}'.format(ssn, ARDT))
        
        if (i <= 1) & (j <= 1):
            caption = """For the four Colorado regions: (row 1) total number of trajectories ran for the {0} season ({1}).
            (row 2) percent of trajectories associated with a landfalling AR based on the 
            {2} ARDT. 
            (row 3--7) same as row 2, broken down by AR scale.""".format(ssn_lbl[i], ssn, ARDT_lbl[j])
        else:
            caption = """ Same as Table S1, but for the {0} season ({1}) and the {2} ARDT.""".format(ssn_lbl[i], ssn, ARDT_lbl[j])
        label = 'table:{0}{1}'.format(ssn, ARDT)
        print(table.to_latex(index=True,
                             caption=caption,
                             float_format="{:0.1f}".format,  # Formats floats to two decimal places
                             bold_rows = True,
                             label=label,
                             position="htbp",  # The preferred positions where the table should be placed in the document ('here', 'top', 'bottom', 'page')
                             column_format="lp{2cm}cccc",  # The format of the columns: left-aligend first column and center-aligned remaining columns as per APA guidelines
                             escape=False,
                            ))


% Table for NDJFMA tARget
\begin{table}[htbp]
\caption{For the four Colorado regions: (row 1) total number of trajectories ran for the cool season (NDJFMA).
            (row 2) percent of trajectories associated with a landfalling AR based on the 
            tARgetv4 ARDT. 
            (row 3--7) same as row 2, broken down by AR scale.}
\label{table:NDJFMAtARget}
\begin{tabular}{lp{2cm}cccc}
\toprule
 & Northwestern & Southwestern & Rio Grande & Eastern \\
\midrule
\textbf{Total Trajectories} & 2118 & 2263 & 848 & 6264 \\
\textbf{Landfalling AR Trajectories} & 20.6\% & 32.9\% & 30.0\% & 2.7\% \\
\textbf{AR scale 1} & 10.6\% & 15.5\% & 13.9\% & 1.4\% \\
\textbf{AR scale 2} & 5.9\% & 9.9\% & 8.4\% & 0.8\% \\
\textbf{AR scale 3} & 1.8\% & 3.6\% & 3.3\% & 0.2\% \\
\textbf{AR scale 4} & 0.2\% & 0.5\% & 0.8\% & 0.0\% \\
\textbf{AR scale 5} & 0.1\% & 0.0\% & 0.0\% & 0.0\% \\
\bottomrule
\end{tabular}
\end{table}

% Table for NDJFMA ar
\begin{table}[htbp]
\caption{For the four Colorado region